In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import MaxNLocator
import os

# ============================================================
# Load Figure 1 data from GitHub repository
# ============================================================
data_path = os.path.join("data", "Figure_1.npz")
data = np.load(data_path, allow_pickle=True)

# ---------------- Panel (a) ----------------
eta_vals = data["eta_vals"]
delta_vals = data["delta_vals"]
alpha_minuss = data["alpha_minus"]
omega_r = data["omega_r"]

# ---------------- Panel (b) ----------------
a_minus_ss = data["a_minus_ss"]
b_plus_ss = data["b_plus_ss"]
Nop_ss = data["Nop_ss"]

# ---------------- Panels (c,d,e) ----------------
X = data["X"]
Y = data["Y"]
snapshots = data["snapshots"]
eta_list = data["eta_list"]

# ============================================================
# Global plotting style (APS style)
# ============================================================
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 12,
    "xtick.direction": "out",
    "ytick.direction": "out",
})

# ============================================================
# Main figure layout
# ============================================================
fig = plt.figure(figsize=(7, 5.5))
gs = GridSpec(
    2, 3,
    height_ratios=[1.6, 1.4],
    hspace=0.5,
    wspace=0.25
)

# ============================================================
# Top row split into 2 panels
# ============================================================
gs_top = gs[0, :].subgridspec(1, 2, wspace=0.50)

# ============================================================
# (a) Contour plot
# ============================================================
ax_a = fig.add_subplot(gs_top[0, 0])

Eta, Delta = np.meshgrid(
    eta_vals / omega_r,
    delta_vals / omega_r,
    indexing='ij'
)

c = ax_a.contourf(
    Eta,
    Delta,
    np.abs(alpha_minuss),
    levels=300,
    cmap="Blues"
)

ax_a.set_xlabel(r"$\eta/\omega_r$")
ax_a.set_ylabel(r"$\delta_c/\omega_r$")
ax_a.set_title(r"(a)\, $|\alpha_-|, |\beta_+|$", fontsize=14, pad=6)

cbar_a = fig.colorbar(c, ax=ax_a, pad=0.02)
cbar_a.locator = MaxNLocator(nbins=3)
cbar_a.update_ticks()
cbar_a.set_ticks([0, 0.5, 1])
cbar_a.ax.tick_params(length=4)

# ============================================================
# (b) Steady-state amplitudes
# ============================================================
ax_b = fig.add_subplot(gs_top[0, 1])

ax_b.plot(
    eta_vals / omega_r,
    a_minus_ss,
    lw=1.5,
    color='tab:green',
    label=r'$|\alpha_-|$'
)

ax_b.plot(
    eta_vals / omega_r,
    b_plus_ss,
    lw=1.5,
    color='tab:orange',
    ls='--',
    label=r'$|\beta_+|$'
)

ax_b.plot(
    eta_vals / omega_r,
    Nop_ss,
    lw=1.5,
    color='tab:blue',
    label=r'$|\mathcal{N}|$'
)

ax_b.set_xlabel(r'$\eta / \omega_r$')
ax_b.set_ylabel('Amplitude')
ax_b.set_ylim([-0.05, 1.1])
ax_b.set_xlim([0, 100])

ax_b.legend(
    frameon=False,
    loc='upper left',
    bbox_to_anchor=(0.0, 1.06),
    handlelength=0.9
)

ax_b.set_title(r"(b)", fontsize=14, pad=6)

# ============================================================
# Bottom row ring density plots
# ============================================================
axes_bottom = []
panel_labels = ['(c)', '(d)', '(e)']

vmin = np.nanmin(snapshots)
vmax = np.nanmax(snapshots)

for j, (eta, lab) in enumerate(zip(eta_list, panel_labels)):

    ax = fig.add_subplot(gs[1, j])
    axes_bottom.append(ax)

    cmap = plt.cm.inferno.copy()
    cmap.set_bad(alpha=0)

    im = ax.pcolormesh(
        X * 1e6,
        Y * 1e6,
        snapshots[j],
        cmap=cmap,
        shading='auto',
        vmin=vmin,
        vmax=vmax
    )

    ax.set_facecolor('black')
    ax.set_aspect('equal')

    ax.set_xticks([-10, 0, 10])
    ax.set_yticks([-10, 0, 10])

    ax.set_title(
        f"{lab} $\\eta = {int(eta/omega_r)}\\,\\omega_r$",
        fontsize=14
    )

    ax.set_xlabel(r'$x\,(\mu\mathrm{m})$')

    if j == 0:
        ax.set_ylabel(r'$y\,(\mu\mathrm{m})$')
    else:
        ax.set_yticklabels([])

# ============================================================
# Shared colorbar for bottom row
# ============================================================
cbar = fig.colorbar(
    im,
    ax=axes_bottom,
    fraction=0.045,
    shrink=0.92,
    pad=0.01
)

cbar.set_label(r'$|\Psi(\phi)|^2$', fontsize=14)
cbar.ax.tick_params(length=4)

plt.show()